In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

base_path = "/Volumes/dlt_hc_fraud_detection/prod_schema_hc/hc_volume_2026"


claim_lines_schema = StructType([
    StructField("claim_id", IntegerType()),
    StructField("procedures",
        ArrayType(
            StructType([
                StructField("code", StringType()),
                StructField("charge", DoubleType())
            ])
        )
    )
])


rules_schema = StructType([
    StructField("rule_id", IntegerType()),
    StructField("rule", StringType())
])


audit_schema = StructType([
    StructField("log_id", IntegerType()),
    StructField("claim_id", IntegerType()),
    StructField("event", StringType()),
    StructField("timestamp", StringType())
])


def read_csv_stream(path):

    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("header","true")
        .option("inferSchema","true")
        .load(path)
        .withColumn("ingestion_ts",current_timestamp())
    )


def read_json_stream(path,schema):

    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","json")
        .option("multiLine","true")
        .schema(schema)
        .load(path)
        .withColumn("ingestion_ts",current_timestamp())
    )


@dlt.table
def bronze_claims():

    df = read_csv_stream(f"{base_path}/HC_2026_claims")

    return df.select(
        col("claim_id").cast("int"),
        col("provider_id").cast("int"),
        col("patient_id").cast("int"),
        col("claim_amount").cast("double"),
        col("claim_date"),
        "ingestion_ts"
    )


@dlt.table
def bronze_payments():

    df = read_csv_stream(f"{base_path}/HC_2026_payments")

    return df.select(
        when(col("payment_id").isNull(),
             concat(lit("PMT_"),col("claim_id"))
        ).otherwise(col("payment_id")).alias("payment_id"),

        col("claim_id").cast("int"),
        col("paid_amount").cast("double"),
        "_rescued_data",
        "ingestion_ts"
    )


@dlt.table
def bronze_providers():
    return read_csv_stream(f"{base_path}/HC_2026_providers")


@dlt.table
def bronze_patients():
    return read_csv_stream(f"{base_path}/HC_2026_patients")


@dlt.table
def bronze_claim_lines():

    return read_json_stream(
        f"{base_path}/HC_2026_claim_lines",
        claim_lines_schema
    )


@dlt.table
def bronze_rules():
    return read_json_stream(
        f"{base_path}/HC_2026_rules",
        rules_schema
    )


@dlt.table
def bronze_audit_logs():
    return read_json_stream(
        f"{base_path}/HC_2026_audit_logs",
        audit_schema
    )

In [0]:
dlt.create_streaming_table("silver_claims")

dlt.apply_changes(
    target="silver_claims",
    source="bronze_claims",
    keys=["claim_id"],
    sequence_by=col("ingestion_ts"),
    stored_as_scd_type=2
)


dlt.create_streaming_table("silver_providers")

dlt.apply_changes(
    target="silver_providers",
    source="bronze_providers",
    keys=["provider_id"],
    sequence_by=col("ingestion_ts"),
    stored_as_scd_type=2
)


dlt.create_streaming_table("silver_patients")

dlt.apply_changes(
    target="silver_patients",
    source="bronze_patients",
    keys=["patient_id"],
    sequence_by=col("ingestion_ts"),
    stored_as_scd_type=2
)


@dlt.table
def silver_payments():

    return (
        dlt.read_stream("bronze_payments")
        .select(
            "payment_id",
            "claim_id",
            "paid_amount",
            "ingestion_ts"
        )
    )


@dlt.table
def silver_claim_lines():

    return (
        dlt.read_stream("bronze_claim_lines")
        .select(
            "claim_id",
            explode("procedures").alias("proc"),
            "ingestion_ts"
        )
        .select(
            "claim_id",
            col("proc.code").alias("procedure_code"),
            col("proc.charge").alias("charge"),
            "ingestion_ts"
        )
    )


@dlt.table
def silver_rules():
    return dlt.read_stream("bronze_rules")


@dlt.table
def silver_audit_logs():

    return (
        dlt.read_stream("bronze_audit_logs")
        .withColumn("event_ts",to_timestamp("timestamp"))
        .drop("timestamp")
    )

In [0]:
@dlt.table
def dim_provider():

    return (
        dlt.read("silver_providers")
        .filter(col("__END_AT").isNull())
        .select(
            "provider_id",
            col("name").alias("provider_name"),
            "specialty",
            "risk_score"
        )
    )


@dlt.table
def dim_patient():

    return (
        dlt.read("silver_patients")
        .filter(col("__END_AT").isNull())
        .select("patient_id")
    )


@dlt.table
def dim_date():

    return (
        dlt.read("silver_claims")
        .filter(col("__END_AT").isNull())
        .select("claim_date")
        .distinct()
        .withColumn("year", year("claim_date"))
        .withColumn("month", month("claim_date"))
        .withColumn("day", dayofmonth("claim_date"))
    )


@dlt.table
def fact_fraud_claims():

    # FIX: read silver tables in batch mode
    claims = (
        dlt.read("silver_claims")
        .filter(col("__END_AT").isNull())
    )

    payments = dlt.read("silver_payments")

    providers = (
        dlt.read("silver_providers")
        .filter(col("__END_AT").isNull())
    )

    claim_lines = dlt.read("silver_claim_lines")


    proc_agg = (
        claim_lines
        .groupBy("claim_id")
        .agg(sum("charge").alias("total_procedure_charge"))
    )


    df = (
        claims
        .join(payments, "claim_id", "left")
        .join(proc_agg, "claim_id", "left")
        .join(providers, "provider_id", "left")
        .fillna({
            "paid_amount": 0,
            "total_procedure_charge": 0,
            "risk_score": 0
        })
    )


    df = df.withColumn(
        "payment_anomaly_flag",
        when(col("paid_amount") > col("claim_amount"), 1).otherwise(0)
    )


    df = df.withColumn(
        "high_claim_flag",
        when(col("claim_amount") >= 75000, 1).otherwise(0)
    )


    df = df.withColumn(
        "procedure_mismatch_flag",
        when(col("total_procedure_charge") > col("claim_amount"), 1).otherwise(0)
    )


    df = df.withColumn(
        "fraud_score",
        col("payment_anomaly_flag") * 40 +
        col("high_claim_flag") * 20 +
        col("procedure_mismatch_flag") * 20 +
        when(col("risk_score") >= 0.75, 30)
        .when(col("risk_score") >= 0.45, 15)
        .otherwise(0)
    )


    df = df.withColumn(
        "fraud_flag",
        when(col("fraud_score") >= 70, "YES")
        .when(col("fraud_score") >= 40, "REVIEW")
        .otherwise("NO")
    )


    df = df.withColumn(
        "fraud_severity",
        when(col("fraud_score") >= 70, "HIGH")
        .when(col("fraud_score") >= 40, "MEDIUM")
        .otherwise("LOW")
    )


    return df.select(
        "claim_id",
        "provider_id",
        "patient_id",
        "claim_date",
        "claim_amount",
        "paid_amount",
        "total_procedure_charge",
        "payment_anomaly_flag",
        "high_claim_flag",
        "procedure_mismatch_flag",
        "fraud_score",
        "fraud_flag",
        "fraud_severity"
    )